In [ ]:
# Create and validate our Spark session
from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .appName("projet-final") \
        .master("local[*]") \
        .getOrCreate()

print("Spark Version:", spark.version)

In [ ]:
# Test the connection to the HDFS
from schemas import TABLES_SCHEMAS
schema = TABLES_SCHEMAS["olist_orders_dataset"]

hdfs_root_path = "hdfs://namenode:9000/user/mti850/"
file_name = "olist_orders_dataset.csv"

df_raw = spark.read.format("csv") \
            .option("header", True) \
            .schema(schema) \
            .load(f"{hdfs_root_path}raw/{file_name}")

df_raw.show(3)

#### Save the Dataframe in Parquet format


Note: If you get an AccessControlException: Permission denied error you'll need to give write permissions to the folder.

In [ ]:
# Loop to convert all csv files into Parquet

for table_name, schema in TABLES_SCHEMAS.items():
    csv_path = f"{hdfs_root_path}/raw/{table_name}.csv"
    parquet_path = f"{hdfs_root_path}/processed/{table_name}.parquet"

    df = (
        spark.read.format("csv")
        .option("header", True)
        .schema(schema)
        .load(csv_path)
    )
    
    df.write.mode("overwrite").option("compression", "snappy").parquet(
        parquet_path
    )
    
    # Validation
    row_count = df.count()
    print(f"[OK] {table_name:35} -> {row_count:>7} line written.")


Commande: 

*hdfs dfs -du -s -h /user/mti850/processed/olist_orders_dataset.parquet*

Taille sur disque (net)	Reduction
 CSV brut (Ligne) - olist_orders_dataset.csv 16.8 Mo
#Parquet - olist_orders_dataset.parquet	6.4 Mo

Reduction = 61.9 %

In [ ]:
# Test de relecture
df_orders_parquet = spark.read. \
        parquet("hdfs://namenode:9000/user/mti850/processed/olist_orders_dataset.parquet")

df_orders_parquet.printSchema()

In [ ]:
spark.stop()